# Full-Scale TruthfulQA Runner (Updated)

This notebook includes everything required: setup, Ollama install/start, model pulls, full-study run, evaluation, plots, publication artifacts, export, and persistent checkpointing.

- Set `RUN_STRONG_MACHINE = True` for **all 3 models** (`llama3.2`, `mistral`, `llama3.3:70b`).
- Set `RUN_STRONG_MACHINE = False` for only 2 models (`llama3.2`, `mistral`).
- The notebook checkpoints outputs to **Google Drive** after major stages so Colab disconnects do not force a full rerun.
- A recovery cell is included to restore saved outputs after a runtime reset.

In [ ]:
# Clean, deterministic setup (prevents nested clone paths)
%cd /content
!rm -rf llm-hallucination-phoenix
!git clone https://github.com/sahelmain/llm-hallucination-phoenix.git
%cd /content/llm-hallucination-phoenix

# Ensure exact latest remote state
!git fetch --all
!git reset --hard origin/main
!git clean -fd

# Ensure output folders always exist
!mkdir -p data reports snapshots

# Patch runner scripts to use more parallel workers (GPU can handle it)
import pathlib
GPU_NUM_WORKERS = 8
for script in ["src/run_experiment.py", "src/run_round2_matrix.py", "src/evaluate_metrics.py"]:
    p = pathlib.Path(script)
    if p.exists():
        text = p.read_text()
        if "NUM_WORKERS = 4" in text:
            p.write_text(text.replace("NUM_WORKERS = 4", f"NUM_WORKERS = {GPU_NUM_WORKERS}"))
            print(f"Patched {script}: NUM_WORKERS -> {GPU_NUM_WORKERS}")

# Quick sanity check
!pwd
!ls

!pip install -q -r requirements.txt

In [ ]:
# Mount Google Drive and create persistent checkpoint folders
from google.colab import drive
from pathlib import Path


drive.mount("/content/drive", force_remount=False)

DRIVE_CHECKPOINT_ROOT = Path("/content/drive/MyDrive/llm_hallucination_phoenix_checkpoints")
DRIVE_DATA_DIR = DRIVE_CHECKPOINT_ROOT / "data"
DRIVE_REPORTS_DIR = DRIVE_CHECKPOINT_ROOT / "reports"
DRIVE_SNAPSHOT_DIR = DRIVE_CHECKPOINT_ROOT / "snapshots"

for path in [DRIVE_CHECKPOINT_ROOT, DRIVE_DATA_DIR, DRIVE_REPORTS_DIR, DRIVE_SNAPSHOT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Drive checkpoint root:", DRIVE_CHECKPOINT_ROOT)
print("Drive data dir:", DRIVE_DATA_DIR)
print("Drive reports dir:", DRIVE_REPORTS_DIR)
print("Drive snapshots dir:", DRIVE_SNAPSHOT_DIR)

In [ ]:
# Helper functions for Drive syncs and milestone zip snapshots
import shutil
import time
import zipfile
from pathlib import Path

REPO_ROOT = Path.cwd()
LOCAL_SNAPSHOT_DIR = REPO_ROOT / "snapshots"
LOCAL_SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)


def _resolve_checkpoint_files(patterns=None, extra_paths=None):
    files = []
    for pattern in patterns or []:
        files.extend(path for path in REPO_ROOT.glob(pattern) if path.is_file())
    for raw_path in extra_paths or []:
        path = Path(raw_path)
        if not path.is_absolute():
            path = REPO_ROOT / path
        if path.exists() and path.is_file():
            files.append(path)
    deduped = []
    seen = set()
    for path in files:
        resolved = path.resolve()
        if resolved not in seen:
            deduped.append(path)
            seen.add(resolved)
    return deduped


def checkpoint_stage(stage_name, patterns=None, extra_paths=None, zip_snapshot=True):
    files_to_save = _resolve_checkpoint_files(patterns=patterns, extra_paths=extra_paths)
    if not files_to_save:
        print(f"No files found for checkpoint stage: {stage_name}")
        return []

    copied = []
    for source in files_to_save:
        relative = source.relative_to(REPO_ROOT)
        destination = DRIVE_CHECKPOINT_ROOT / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
        copied.append(str(relative))

    if zip_snapshot:
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        local_zip = LOCAL_SNAPSHOT_DIR / f"{stage_name}_{timestamp}.zip"
        with zipfile.ZipFile(local_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
            for source in files_to_save:
                archive.write(source, source.relative_to(REPO_ROOT))
        shutil.copy2(local_zip, DRIVE_SNAPSHOT_DIR / local_zip.name)
        print("Created local snapshot:", local_zip)
        print("Copied snapshot to Drive:", DRIVE_SNAPSHOT_DIR / local_zip.name)

    print(f"Checkpoint '{stage_name}' saved {len(copied)} files.")
    for item in copied:
        print(" -", item)
    return copied

In [ ]:
# Optional recovery cell: restore saved outputs from Drive after a runtime reset
import shutil

for folder_name in ["data", "reports"]:
    source_dir = DRIVE_CHECKPOINT_ROOT / folder_name
    destination_dir = REPO_ROOT / folder_name
    destination_dir.mkdir(parents=True, exist_ok=True)

    restored_count = 0
    if source_dir.exists():
        for source in source_dir.rglob("*"):
            if source.is_file():
                relative = source.relative_to(DRIVE_CHECKPOINT_ROOT)
                destination = REPO_ROOT / relative
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(source, destination)
                restored_count += 1

    print(f"Restored {restored_count} files from {source_dir} to {destination_dir}")

print("If you are resuming from restored outputs, set CLEAN_RERUN = False before running the preflight cell, then skip the generation cell and continue with evaluation/plots/export as needed.")

In [ ]:
# Install and start Ollama with full GPU utilization
import os
import subprocess
import time

!curl -fsSL https://ollama.com/install.sh | sh

print("--- GPU check ---")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

os.environ["OLLAMA_NUM_PARALLEL"] = "4"
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"
os.environ["OLLAMA_KEEP_ALIVE"] = "30m"

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=os.environ.copy(),
)
time.sleep(5)
!ollama --version
!ollama list
print("\nOLLAMA_NUM_PARALLEL:", os.environ.get("OLLAMA_NUM_PARALLEL"))
print("OLLAMA_FLASH_ATTENTION:", os.environ.get("OLLAMA_FLASH_ATTENTION"))

In [ ]:
# Choose mode here
RUN_STRONG_MACHINE = True
print("RUN_STRONG_MACHINE =", RUN_STRONG_MACHINE)

In [ ]:
# Pull models based on machine strength
if RUN_STRONG_MACHINE:
    !ollama pull llama3.2
    !ollama pull mistral
    !ollama pull llama3.3:70b
else:
    !ollama pull llama3.2
    !ollama pull mistral

!ollama list

In [ ]:
# Update config to match selected mode (models + templates)
import yaml

with open("config/experiment.yaml") as f:
    cfg = yaml.safe_load(f)

if RUN_STRONG_MACHINE:
    cfg["models"]["full_study_models"] = [
        {"provider": "ollama", "model_name": "llama3.2", "temperature": 0.0, "max_tokens": 128},
        {"provider": "ollama", "model_name": "mistral", "temperature": 0.0, "max_tokens": 128},
        {"provider": "ollama", "model_name": "llama3.3:70b", "temperature": 0.0, "max_tokens": 128},
    ]
else:
    cfg["models"]["full_study_models"] = [
        {"provider": "ollama", "model_name": "llama3.2", "temperature": 0.0, "max_tokens": 128},
        {"provider": "ollama", "model_name": "mistral", "temperature": 0.0, "max_tokens": 128},
    ]

# Always run the expanded 4-template full study matrix
cfg["prompt_templates"]["full_study"] = [
    "factual_direct",
    "strict_abstain",
    "chain_of_thought",
    "concise_factual",
]

cfg.setdefault("execution", {}).setdefault("full_study", {})["max_workers"] = GPU_NUM_WORKERS

with open("config/experiment.yaml", "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print("Configured models:", [m["model_name"] for m in cfg["models"]["full_study_models"]])
print("Configured full-study templates:", cfg["prompt_templates"]["full_study"])
print("Configured max_workers:", GPU_NUM_WORKERS)

In [ ]:
# Optional clean rerun: remove old outputs before running.
# Set to False if you want resume behavior.
CLEAN_RERUN = True

if CLEAN_RERUN:
    !rm -f data/full_study_results.csv \
          data/experiment_results.csv \
          data/evaluated_results_full.csv \
          data/evaluated_results.csv \
          data/metrics_full.json \
          data/metrics.json \
          data/category_metrics_full.csv \
          data/category_why_signals_full.csv \
          data/paired_comparisons_full.json \
          data/paired_comparisons_by_category_full.json \
          data/table_top10_vulnerable_categories_full.csv \
          data/table_model_category_hallucination_full.csv \
          data/table_top_failure_reason_by_model_category_full.csv \
          reports/full_reproducibility_appendix.md
    print("Old outputs removed. Fresh rerun enabled.")
else:
    print("Keeping existing outputs. Resume mode enabled.")

# Preflight checks (fail fast if repo/path/config are wrong)
import os
import yaml

required_scripts = [
    "src/evaluate_metrics.py",
    "src/generate_plots.py",
]
for path in required_scripts:
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Missing required script: {path}")

for folder in ["data", "reports"]:
    os.makedirs(folder, exist_ok=True)

has_modern_runner = os.path.isfile("src/run_round2_matrix.py") and "--mode" in open("src/run_round2_matrix.py").read()
has_experiment_runner = os.path.isfile("src/run_experiment.py")
has_artifact_script = os.path.isfile("src/generate_full_study_artifacts.py")

if has_modern_runner:
    RUN_COMMAND = "python -u src/run_round2_matrix.py --mode full --disable-phoenix"
    PRIMARY_RESULT_FILE = "data/full_study_results.csv"
elif has_experiment_runner:
    RUN_COMMAND = "python -u src/run_experiment.py"
    PRIMARY_RESULT_FILE = "data/experiment_results.csv"
else:
    raise FileNotFoundError("No supported runner found (need run_round2_matrix.py with --mode or run_experiment.py).")

with open("config/experiment.yaml") as f:
    check_cfg = yaml.safe_load(f)

patched_legacy_keys = False
check_cfg.setdefault("models", {})
check_cfg.setdefault("dataset", {})
if "experiment_models" not in check_cfg["models"] and "full_study_models" in check_cfg["models"]:
    check_cfg["models"]["experiment_models"] = check_cfg["models"]["full_study_models"]
    patched_legacy_keys = True
if "sample_size" not in check_cfg["dataset"] and "full_sample_size" in check_cfg["dataset"]:
    check_cfg["dataset"]["sample_size"] = check_cfg["dataset"]["full_sample_size"]
    patched_legacy_keys = True

if not has_modern_runner and has_experiment_runner:
    pt = check_cfg.get("prompt_templates", {})
    if isinstance(pt, dict):
        flat_templates = pt.get("full_study") or pt.get("round2") or []
        check_cfg["prompt_templates"] = flat_templates
        patched_legacy_keys = True
        print("Flattened prompt_templates for legacy runner:", flat_templates)
    eval_section = check_cfg.setdefault("evaluation", {})
    if "repetitions_per_item" not in eval_section:
        eval_section["repetitions_per_item"] = eval_section.get("full_study_repetitions_per_item", 1)
        patched_legacy_keys = True

if patched_legacy_keys:
    with open("config/experiment.yaml", "w") as f:
        yaml.dump(check_cfg, f, default_flow_style=False)
    print("Patched config/experiment.yaml for legacy runner compatibility.")

if "full_study_models" in check_cfg.get("models", {}):
    model_cfg = check_cfg["models"]["full_study_models"]
elif "experiment_models" in check_cfg.get("models", {}):
    model_cfg = check_cfg["models"]["experiment_models"]
else:
    raise ValueError("Config missing full_study_models or experiment_models")

pt_cfg = check_cfg.get("prompt_templates", {})
if isinstance(pt_cfg, list):
    template_cfg = pt_cfg
elif isinstance(pt_cfg, dict) and "full_study" in pt_cfg:
    template_cfg = pt_cfg["full_study"]
elif isinstance(pt_cfg, dict):
    template_cfg = next((v for v in pt_cfg.values() if isinstance(v, list)), [])
else:
    raise ValueError("Config missing prompt templates")

n_items = check_cfg["dataset"].get("full_sample_size", check_cfg["dataset"].get("sample_size", 817))
n_models = len(model_cfg)
n_templates = len(template_cfg)
n_prompt_types = 2
n_reps = check_cfg.get("evaluation", {}).get("full_study_repetitions_per_item", check_cfg.get("evaluation", {}).get("repetitions_per_item", 1))
expected_full_rows = n_items * n_models * n_templates * n_prompt_types * n_reps

print("Preflight OK")
print("cwd:", os.getcwd())
print("run_command:", RUN_COMMAND)
print("primary_result_file:", PRIMARY_RESULT_FILE)
print("has_artifact_script:", has_artifact_script)
print("models:", [m.get("model_name", "?") for m in model_cfg])
print("templates:", template_cfg)
print("expected_full_rows:", expected_full_rows)

In [ ]:
# Long-running generation. Rerun safely (resume/checkpoint supported).
import os
import shutil
import subprocess
import time

print("Executing:", RUN_COMMAND)
checkpoint_interval_seconds = 300
proc = subprocess.Popen(
    RUN_COMMAND,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
last_checkpoint_time = time.time()

for line in proc.stdout:
    print(line, end="")
    now = time.time()
    if now - last_checkpoint_time >= checkpoint_interval_seconds:
        partial_files = [
            path
            for path in ["data/full_study_results.csv", "data/experiment_results.csv"]
            if os.path.exists(path)
        ]
        if partial_files:
            checkpoint_stage("generation_partial", extra_paths=partial_files)
            last_checkpoint_time = now
            print("\nCheckpointed partial generation outputs to Drive.")

ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"Generation failed with exit code {ret}")

# Normalize output name for downstream cells
if os.path.exists("data/experiment_results.csv") and not os.path.exists("data/full_study_results.csv"):
    shutil.copy("data/experiment_results.csv", "data/full_study_results.csv")
    print("Copied experiment_results.csv -> full_study_results.csv")

checkpoint_stage(
    "generation_complete",
    extra_paths=["data/full_study_results.csv", "data/experiment_results.csv"],
)
print("Generation complete and checkpointed.")

In [ ]:
# Evaluate metrics + category analyses
import os
import shutil
import subprocess
import yaml

with open("config/experiment.yaml") as f:
    eval_cfg = yaml.safe_load(f)

patched_eval_keys = False
available_models = eval_cfg.get("models", {}).get("full_study_models") or eval_cfg.get("models", {}).get("experiment_models", [])
if "judge_model" not in eval_cfg.get("models", {}) and available_models:
    judge_name = "mistral" if any(model.get("model_name") == "mistral" for model in available_models) else available_models[0].get("model_name")
    eval_cfg.setdefault("models", {})["judge_model"] = {
        "provider": "ollama",
        "model_name": judge_name,
        "temperature": 0.0,
        "max_tokens": 16,
    }
    patched_eval_keys = True
if "evaluators" not in eval_cfg:
    eval_cfg["evaluators"] = {}
    patched_eval_keys = True
if patched_eval_keys:
    with open("config/experiment.yaml", "w") as f:
        yaml.dump(eval_cfg, f, default_flow_style=False)
    print("Patched config/experiment.yaml with legacy evaluation keys.")

if os.path.exists("data/full_study_results.csv") and not os.path.exists("data/experiment_results.csv"):
    shutil.copy("data/full_study_results.csv", "data/experiment_results.csv")
    print("Copied full_study_results.csv -> experiment_results.csv (evaluate_metrics.py expects this name)")

if not os.path.exists("data/experiment_results.csv"):
    raise FileNotFoundError("No result CSV found. Generation (Cell 10) must complete first.")

subprocess.run("python src/evaluate_metrics.py", shell=True, check=True)

evaluation_outputs = [
    "data/evaluated_results.csv",
    "data/evaluated_results_full.csv",
    "data/metrics.json",
    "data/metrics_full.json",
    "data/category_metrics_full.csv",
    "data/category_why_signals_full.csv",
    "data/paired_comparisons_full.json",
    "data/paired_comparisons_by_category_full.json",
]
missing_eval_outputs = [path for path in evaluation_outputs[:4] if not os.path.exists(path)]
if len(missing_eval_outputs) == 4:
    raise FileNotFoundError("Evaluation finished but expected outputs were not created.")

checkpoint_stage("post_evaluation", extra_paths=evaluation_outputs)
print("Evaluation outputs checkpointed.")

In [ ]:
# Generate plots and publication artifacts
import subprocess

subprocess.run("python src/generate_plots.py", shell=True, check=True)

if has_artifact_script:
    subprocess.run("python src/generate_full_study_artifacts.py --suffix full", shell=True, check=True)
    print("Publication artifact script completed.")
else:
    print("Skipping generate_full_study_artifacts.py (not present in this repo version).")

checkpoint_stage(
    "post_plots",
    patterns=["data/*.png", "data/table_*", "data/category_*", "data/paired_*", "reports/*.md"],
    extra_paths=[
        "data/metrics.json",
        "data/metrics_full.json",
        "data/evaluated_results.csv",
        "data/evaluated_results_full.csv",
    ],
)
print("Plot and report outputs checkpointed.")

In [ ]:
# Post-run verification (must pass before export)
import glob
import json
import os
import pandas as pd

if os.path.exists("data/full_study_results.csv"):
    result_file = "data/full_study_results.csv"
elif os.path.exists("data/experiment_results.csv"):
    result_file = "data/experiment_results.csv"
else:
    result_file = PRIMARY_RESULT_FILE

if os.path.exists("data/evaluated_results_full.csv"):
    eval_file = "data/evaluated_results_full.csv"
    metrics_file = "data/metrics_full.json"
else:
    eval_file = "data/evaluated_results.csv"
    metrics_file = "data/metrics.json"

required_files = [result_file, eval_file, metrics_file]
if has_artifact_script:
    required_files.extend([
        "data/category_metrics_full.csv",
        "data/category_why_signals_full.csv",
        "data/paired_comparisons_full.json",
        "data/paired_comparisons_by_category_full.json",
        "data/table_top10_vulnerable_categories_full.csv",
        "data/table_model_category_hallucination_full.csv",
        "data/table_top_failure_reason_by_model_category_full.csv",
        "reports/full_reproducibility_appendix.md",
    ])

missing = [p for p in required_files if not os.path.exists(p)]
if missing:
    raise FileNotFoundError("Missing required outputs:\n" + "\n".join(missing))

df = pd.read_csv(result_file)
calc_expected = (
    df["item_id"].nunique()
    * df["model"].nunique()
    * df["template"].nunique()
    * df["prompt_type"].nunique()
    * df["repetition"].nunique()
)
if len(df) != calc_expected:
    raise RuntimeError(f"Row count mismatch: rows={len(df)} expected={calc_expected}")

if "expected_full_rows" in globals() and len(df) != expected_full_rows:
    raise RuntimeError(f"Config expectation mismatch: rows={len(df)} expected_full_rows={expected_full_rows}")

error_count = int(df["output"].astype(str).str.startswith("[ERROR]").sum())

with open(metrics_file) as f:
    metrics = json.load(f)
pngs = sorted(glob.glob("data/*.png"))

print("Verification PASS")
print("result_file:", result_file)
print("eval_file:", eval_file)
print("metrics_file:", metrics_file)
print("rows:", len(df))
print("models:", sorted(df["model"].unique().tolist()))
print("templates:", sorted(df["template"].unique().tolist()))
print("errors:", error_count)
print("hallucination_rate:", round(metrics["hallucination_rate"]["value"], 6))
print("accuracy:", round(metrics["accuracy"]["value"], 6))
print("consistency:", round(metrics["consistency"]["value"], 6))
print("plot_count:", len(pngs))
if "DRIVE_CHECKPOINT_ROOT" in globals():
    print("drive_checkpoint_root:", DRIVE_CHECKPOINT_ROOT)

In [ ]:
# Export bundle (guarded)
import os
import shutil
import time
from google.colab import files

must_exist = []
if os.path.exists("data/full_study_results.csv"):
    must_exist.append("data/full_study_results.csv")
elif os.path.exists("data/experiment_results.csv"):
    must_exist.append("data/experiment_results.csv")
else:
    must_exist.append(PRIMARY_RESULT_FILE)

if os.path.exists("data/evaluated_results_full.csv"):
    must_exist.append("data/evaluated_results_full.csv")
    must_exist.append("data/metrics_full.json")
else:
    must_exist.append("data/evaluated_results.csv")
    must_exist.append("data/metrics.json")

if has_artifact_script:
    must_exist.append("reports/full_reproducibility_appendix.md")

missing = [p for p in must_exist if not os.path.exists(p)]
if missing:
    raise FileNotFoundError("Cannot export. Missing files:\n" + "\n".join(missing))

checkpoint_stage("final_bundle", patterns=["data/*", "reports/*"])
!zip -r full_study_outputs.zip data/ reports/
!ls -lh full_study_outputs.zip
!unzip -l full_study_outputs.zip | sed -n '1,40p'

bundle_copy_name = f"full_study_outputs_{time.strftime('%Y%m%d_%H%M%S')}.zip"
shutil.copy2("full_study_outputs.zip", DRIVE_SNAPSHOT_DIR / bundle_copy_name)
print("Saved final bundle to Drive:", DRIVE_SNAPSHOT_DIR / bundle_copy_name)
files.download("full_study_outputs.zip")

## Optional: Push results to GitHub for teammates

Set `GH_TOKEN` in Colab Secrets first, then run:

```python
from google.colab import userdata
token = userdata.get('GH_TOKEN')

!git add -A
!git config user.email "sahel@example.com"
!git config user.name "Sahel Azzam"
!git commit -m "Add full-study results from Colab" || true
!git remote set-url origin https://{token}@github.com/sahelmain/llm-hallucination-phoenix.git
!git push origin main
```